<a href="https://colab.research.google.com/github/alifihsan7/kelompok9-image-captioning/blob/main/notebooks/02_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
from pathlib import Path

# 1. Menetapkan asumsi jalur folder proyek di Google Drive
# Menangani variasi penamaan default MyDrive / My Drive di Colab
CANDIDATE_PATHS = [
    Path("/content/drive/MyDrive/Kelompok9"),
    Path("/content/drive/My Drive/Kelompok9"),
]

project_dir = None
for path in CANDIDATE_PATHS:
    if path.is_dir():
        project_dir = path
        break

# 2 & 3. Pemeriksaan keberadaan folder utama dan subfolder src
if project_dir is None:
    # 5. Pesan jika folder utama tidak ditemukan
    print("❌ ERROR: Folder utama 'Kelompok9' tidak ditemukan di Google Drive.")
    print("Pemeriksaan yang perlu dilakukan:")
    print("1. Pastikan mount Google Drive berhasil (`drive.mount('/content/drive')`).")
    print("2. Jika 'Kelompok9' adalah folder bersama (Shared Folder), pastikan sudah menambahkan pintasan ke Drive ('Add shortcut to Drive') pada akun Anda.")
    print("3. Pastikan penulisan nama folder tepat 'Kelompok9' (sensitif huruf besar/kecil).")
else:
    # 2. Menentukan lokasi subfolder src
    src_dir = project_dir / "src"

    # 4. Menampilkan lokasi yang ditemukan
    print(f"✅ Folder utama proyek ditemukan : {project_dir}")

    # 3 & 5. Pemeriksaan ketersediaan subfolder src
    if src_dir.is_dir():
        print(f"✅ Subfolder src ditemukan        : {src_dir}")
        print("\nStatus: Jalur valid. Anda siap menyimpan 'src/model.py'.")
    else:
        print(f"❌ ERROR: Subfolder 'src' tidak ditemukan di dalam {project_dir}.")
        print("Peringatan: Folder baru tidak dibuat secara otomatis.")
        print(f"Silakan buat folder 'src' di Google Drive secara manual pada lokasi: {project_dir}")

✅ Folder utama proyek ditemukan : /content/drive/MyDrive/Kelompok9
✅ Subfolder src ditemukan        : /content/drive/MyDrive/Kelompok9/src

Status: Jalur valid. Anda siap menyimpan 'src/model.py'.


Mengatur Direktori Kerja dan Menyimpan src/model.py

In [3]:
import os
import sys

# Pindah ke direktori utama proyek di Drive yang sudah terverifikasi sebelumnya
os.chdir(project_dir)

# Tambahkan direktori proyek ke sys.path agar impor modul 'src' berjalan lancar
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

print(f"Direktori kerja aktif saat ini: {os.getcwd()}")

Direktori kerja aktif saat ini: /content/drive/MyDrive/Kelompok9


In [4]:
%%writefile src/model.py
"""
src/model.py
Modul arsitektur Decoder LSTM untuk Image Captioning.
Mengikuti spesifikasi Kontrak Teknis Data V1 (Kelompok 9).
"""

import torch
import torch.nn as nn


class CaptionDecoder(nn.Module):
    """
    Decoder LSTM untuk Image Captioning.

    Arsitektur:
        - Proyeksi linear fitur visual [B, feature_dim] ke status awal LSTM (h0, c0).
        - Lapisan embedding token kata dengan penanganan padding_idx.
        - LSTM 1 lapis (batch_first=True) untuk pemrosesan sekuens teks.
        - Regulerisasi Dropout pada representasi status tersembunyi.
        - Lapisan linear proyeksi akhir ke dimensi kosakata (raw logits).

    Args:
        vocab_size (int): Jumlah total kosakata unik (V).
        feature_dim (int, optional): Dimensi vektor fitur visual encoder ResNet-50. Default: 2048.
        embed_dim (int, optional): Dimensi representasi vektor kata. Default: 256.
        hidden_size (int, optional): Dimensi ruang status tersembunyi (hidden & cell state). Default: 512.
        dropout_p (float, optional): Probabilitas dropout regulerisasi. Default: 0.5.
        pad_idx (int, optional): Indeks token khusus <pad>. Default: 0.
    """

    def __init__(
        self,
        vocab_size: int,
        feature_dim: int = 2048,
        embed_dim: int = 256,
        hidden_size: int = 512,
        dropout_p: float = 0.5,
        pad_idx: int = 0,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.feature_dim = feature_dim
        self.embed_dim = embed_dim
        self.hidden_size = hidden_size
        self.dropout_p = dropout_p
        self.pad_idx = pad_idx

        # 1. Proyeksi fitur visual ke hidden state (h0) dan cell state (c0) awal
        # [B, feature_dim] -> [B, hidden_size]
        self.init_h = nn.Linear(feature_dim, hidden_size)
        self.init_c = nn.Linear(feature_dim, hidden_size)

        # 2. Lapisan penyematan kata (word embedding)
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx,
        )

        # 3. Lapisan recurrent LSTM 1 lapis
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )

        # 4. Lapisan regulerisasi dropout
        self.dropout = nn.Dropout(p=dropout_p)

        # 5. Lapisan linear keluaran menuju ruang kosakata
        self.fc_out = nn.Linear(hidden_size, vocab_size)

    def forward(
        self, features: torch.Tensor, captions_in: torch.Tensor
    ) -> torch.Tensor:
        """
        Menjalankan satu lintasan komputasi maju menggunakan skema Teacher Forcing.

        Bentuk Tensor:
            - features:    torch.FloatTensor [B, feature_dim]
            - captions_in: torch.LongTensor  [B, L]
            - logits:      torch.FloatTensor [B, L, vocab_size]

        Args:
            features (torch.Tensor): Representasi visual dari encoder berdimensi [B, feature_dim].
            captions_in (torch.Tensor): Sekuens token masukan berdimensi [B, L].

        Returns:
            torch.Tensor: Logits (skor mentah sebelum aktivasi softmax) berdimensi [B, L, vocab_size].
        """
        # 1. Inisialisasi h0 dan c0 dari fitur visual
        # [B, feature_dim] -> [B, hidden_size] -> [1, B, hidden_size]
        h0 = self.init_h(features).unsqueeze(0)
        c0 = self.init_c(features).unsqueeze(0)

        # 2. Pemetaan token teks ke ruang kontinu
        # [B, L] -> [B, L, embed_dim]
        embedded = self.embedding(captions_in)

        # 3. Propagasi sekuensial LSTM
        # lstm_out berdimensi [B, L, hidden_size]
        lstm_out, _ = self.lstm(embedded, (h0, c0))

        # 4. Penerapan dropout pada luaran hidden state
        out = self.dropout(lstm_out)

        # 5. Proyeksi skor mentah ke kosakata tanpa softmax
        # [B, L, hidden_size] -> [B, L, vocab_size]
        logits = self.fc_out(out)

        return logits

Writing src/model.py


Verifikasi Fisik File dan Uji Impor Modul

In [5]:
from pathlib import Path
import importlib

target_file = project_dir / "src" / "model.py"

if target_file.is_file():
    print(f"✅ Berkas berhasil diverifikasi di: {target_file}")

    # Impor kelas CaptionDecoder dari berkas src/model.py
    import src.model
    importlib.reload(src.model)  # Memastikan versi terbaru yang dimuat jika ada perubahan
    from src.model import CaptionDecoder

    print("✅ Berhasil mengimpor 'CaptionDecoder' dari 'src.model'.")
else:
    print(f"❌ ERROR: Berkas {target_file} belum ditemukan di Google Drive.")

✅ Berkas berhasil diverifikasi di: /content/drive/MyDrive/Kelompok9/src/model.py
✅ Berhasil mengimpor 'CaptionDecoder' dari 'src.model'.


Uji Fungsional Tensor Dummy, Loss, dan Aliran Gradien

In [6]:
import torch
import torch.nn as nn

# 1. Parameter tensor pengujian
B = 32
feature_dim = 2048
L = 29
vocab_size = 3000

# 2. Penyiapan tensor dummy
torch.manual_seed(42)
dummy_features = torch.randn(B, feature_dim, dtype=torch.float32)
dummy_captions_in = torch.randint(0, vocab_size, (B, L), dtype=torch.long)
dummy_targets = torch.randint(0, vocab_size, (B, L), dtype=torch.long)

# 3. Inisialisasi model
model = CaptionDecoder(
    vocab_size=vocab_size,
    feature_dim=feature_dim,
    embed_dim=256,
    hidden_size=512,
    dropout_p=0.5,
    pad_idx=0
)

# 4. Uji forward pass
logits = model(dummy_features, dummy_captions_in)
print(f"Bentuk tensor logits   : {list(logits.shape)}")
assert logits.shape == (B, L, vocab_size), f"Dimensi logits tidak cocok! Diharapkan {(B, L, vocab_size)}, didapat {logits.shape}"
print("✅ Bentuk logits sesuai kontrak [B, L, vocab_size].")

# 5. Uji kalkulasi loss (perataan sesuai kontrak: [B*L, V] dan [B*L])
criterion = nn.CrossEntropyLoss(ignore_index=0)
loss = criterion(logits.reshape(-1, vocab_size), dummy_targets.reshape(-1))
print(f"Nilai loss dummy       : {loss.item():.4f}")

# 6. Uji backward pass dan verifikasi gradien
model.zero_grad()
loss.backward()

grad_h = model.init_h.weight.grad
grad_c = model.init_c.weight.grad
grad_fc = model.fc_out.weight.grad

assert grad_h is not None and grad_h.norm().item() > 0, "Gradien pada init_h tidak mengalir!"
assert grad_c is not None and grad_c.norm().item() > 0, "Gradien pada init_c tidak mengalir!"
assert grad_fc is not None and grad_fc.norm().item() > 0, "Gradien pada fc_out tidak mengalir!"

print(f"Norm gradien init_h    : {grad_h.norm().item():.6f} (mengalir)")
print(f"Norm gradien init_c    : {grad_c.norm().item():.6f} (mengalir)")
print(f"Norm gradien fc_out    : {grad_fc.norm().item():.6f} (mengalir)")
print("\n✅ Seluruh pengujian forward pass, perataan loss, dan aliran gradien BERHASIL.")

Bentuk tensor logits   : [32, 29, 3000]
✅ Bentuk logits sesuai kontrak [B, L, vocab_size].
Nilai loss dummy       : 8.0069
Norm gradien init_h    : 0.034732 (mengalir)
Norm gradien init_c    : 0.064243 (mengalir)
Norm gradien fc_out    : 0.116075 (mengalir)

✅ Seluruh pengujian forward pass, perataan loss, dan aliran gradien BERHASIL.
